# T08 — AR Model (AutoRegressive) — v3

**Input:** `data/processed/{train,test}_features.csv`  
**Goal:** predict RUL by forecasting the engine health-index with an AR(p) model until it crosses a failure threshold.

### Method
1. Build a 1-D `health_index` via PCA on rolling-mean sensor features (smoother than raw sensors).
2. Define a failure threshold = median `health_index` of training cycles with `RUL ≤ 5` (near end-of-life).
3. For each test engine:
   - Smooth its health history (rolling median, window=5).
   - Fit `AutoReg(lags=p, trend='c')` on the smoothed series.
   - Forecast up to 150 steps ahead.
   - `RUL` = first forecast step that crosses the threshold (fallback: linear extrapolation).
4. Clip predictions to `[0, 125]`.

AR is used as a **true forecasting model**, not just a denoiser — results depend on the lag order `p`.

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial

from src.models.classical import (
    build_pca_health_index,
    compute_failure_threshold,
    predict_rul_ar,
    predict_dataset,
    simulate_test_from_train,
    RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset

PROC_DIR    = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load processed data & build health index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")
print(f"train: {train.shape}  |  test: {test.shape}")

train, test = build_pca_health_index(train, test, SENSOR_COLS)
print(f"health_index range (train): [{train.health_index.min():.2f}, {train.health_index.max():.2f}]")

## 2. Compute failure threshold from end-of-life training cycles

In [ ]:
THRESHOLD = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print(f"Failure threshold (median health_index at RUL<=5): {THRESHOLD:.3f}")

# sanity check: threshold should sit between healthy and near-failure medians
healthy_med = train.loc[train.RUL >= RUL_CAP - 5, "health_index"].median()
fail_med    = train.loc[train.RUL <= 5,           "health_index"].median()
print(f"  healthy median: {healthy_med:+.2f}   |   near-failure median: {fail_med:+.2f}")
assert healthy_med < THRESHOLD <= fail_med + 1e-6, "threshold looks off"

## 3. Visualize health trajectories for 4 FD001 engines

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
sample = train[train.dataset_id == 1].engine_id.unique()[:4]
for ax, eid in zip(axes, sample):
    g = train[train.engine_id == eid].sort_values("cycle")
    ax.plot(g.cycle, g.health_index, color="steelblue", lw=1, label="health_index")
    ax.axhline(THRESHOLD, color="red", ls="--", label=f"threshold={THRESHOLD:.2f}")
    ax2 = ax.twinx()
    ax2.plot(g.cycle, g.RUL, color="coral", ls=":", alpha=0.6, label="RUL")
    ax.set_title(f"Engine {eid}")
    ax.set_xlabel("cycle"); ax.set_ylabel("health_index", color="steelblue")
    ax2.set_ylabel("RUL", color="coral")
    ax.legend(loc="upper left", fontsize=8)
plt.suptitle("Health index rises toward threshold as RUL → 0", y=1.02)
plt.tight_layout(); plt.show()

## 4. Tune lag order on a simulated validation set

We truncate training engines at random cutoffs to build a synthetic test set with known true RUL.
This keeps the real test set untouched until the final evaluation.

In [ ]:
val_df = simulate_test_from_train(train, cutoff_fraction=0.6, max_engines=150, random_seed=42)
print(f"simulated-val engines: {val_df.engine_id.nunique()}  (rows: {len(val_df)})")

rows = []
for lags in [3, 5, 10, 15, 20]:
    fn = partial(predict_rul_ar, threshold=THRESHOLD, lags=lags)
    yt, yp, _ = predict_dataset(val_df, fn)
    m = evaluate(yt, yp, model_name=f"AR({lags})", verbose=False)
    rows.append({"lags": lags, **m})

tune_df = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
print(tune_df.to_string(index=False))

BEST_LAGS = int(tune_df.iloc[0].lags)
print(f"\nBest lag order: {BEST_LAGS}")

## 5. Final evaluation on official test set

In [ ]:
predict_fn = partial(predict_rul_ar, threshold=THRESHOLD, lags=BEST_LAGS)
y_true, y_pred, dids = predict_dataset(test, predict_fn)

print(f"=== AR({BEST_LAGS}) — Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name="AR")
display(test_metrics)

## 6. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_true, y_pred, alpha=0.4, s=15, color="steelblue")
axes[0].plot([0, RUL_CAP], [0, RUL_CAP], "r--", label="perfect")
axes[0].set_xlabel("true RUL"); axes[0].set_ylabel("predicted RUL")
axes[0].set_title(f"AR({BEST_LAGS}) — Pred vs True"); axes[0].legend()

err = y_pred - y_true
axes[1].hist(err, bins=40, color="coral", edgecolor="white")
axes[1].axvline(0, color="black", ls="--")
axes[1].axvline(err.mean(), color="red", label=f"mean={err.mean():+.1f}")
axes[1].set_xlabel("error (pred-true)  |  + = late"); axes[1].set_title("error distribution"); axes[1].legend()

axes[2].hist(y_pred, bins=40, color="mediumpurple", edgecolor="white")
axes[2].set_xlabel("predicted RUL"); axes[2].set_title("prediction distribution")
n_cap = int((y_pred >= RUL_CAP).sum())
print(f"engines predicted at cap: {n_cap}/{len(y_pred)}")
plt.tight_layout(); plt.show()

## 7. Save predictions

In [ ]:
out = pd.DataFrame({
    "model":      "AR",
    "lags":       BEST_LAGS,
    "y_true":     y_true,
    "y_pred":     y_pred,
    "dataset_id": dids,
})
out.to_csv(RESULTS_DIR / "T08_AR_predictions.csv", index=False)

overall = evaluate(y_true, y_pred, model_name="AR_test", verbose=False)
print(f"saved | RMSE={overall['rmse']:.4f}  NASA={overall['nasa_score']:.2f}")